# FoncierAI — Entraînement des modèles (Régression Linéaire, Random Forest, Isolation Forest)

Ce notebook charge `DATA_final_mercuriale_corrige.xlsx`, nettoie les données, entraîne les 3 modèles utilisés dans l'application, puis les exporte en `.pkl` (téléchargeables depuis l'onglet Output de Kaggle).

## 1. Chargement des données

Sur Kaggle : clique sur **+ Add Input** (à droite) → cherche le dataset que tu as uploadé → il apparaîtra sous `/kaggle/input/<nom-du-dataset>/`.
Adapte le chemin ci-dessous si besoin (vérifie-le avec `!ls /kaggle/input/`).

In [1]:
import pandas as pd
import numpy as np

!ls /kaggle/input/

# Adapte le chemin exact selon le nom donné à ton dataset Kaggle
CHEMIN_FICHIER = "/kaggle/input/foncierai-mercuriale/DATA_final_mercuriale.xlsx"

df = pd.read_excel(CHEMIN_FICHIER, sheet_name="Feuil1")
print(df.shape)
df.head()

'ls' is not recognized as an internal or external command,
operable program or batch file.


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/foncierai-mercuriale/DATA_final_mercuriale.xlsx'

## 2. Nettoyage des données

In [ ]:
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

# Colonnes utiles pour l'entraînement
colonnes_utiles = ["LOTISSEMENT", "SUPERFICIE_M2", "usage", "PRIX-VENTE", "FRAUDE"]
df = df.dropna(subset=["SUPERFICIE_M2", "LOTISSEMENT", "PRIX-VENTE"]).copy()

# FRAUDE manquant -> on considère 0 (non frauduleux) par défaut
df["FRAUDE"] = df["FRAUDE"].fillna(0)

# Normalisation du nom de zone (minuscule, sans espaces superflus)
df["LOTISSEMENT"] = df["LOTISSEMENT"].str.strip().str.lower()

# Infrastructure : le dataset actuel n'a pas de colonne dédiée.
# On l'approxime à partir du TYPE-DOCUMENTS (certificat = zone mieux équipée) --
# À AJUSTER si tu as une vraie colonne infrastructure, sinon ce proxy reste grossier.
df["infrastructure"] = df["TYPE-DOCUMENTS"].str.contains("certificat", case=False, na=False).astype(int)

print("Zones après nettoyage :", df["LOTISSEMENT"].value_counts())
print("Lignes restantes :", len(df))

## 3. Encodage et préparation des features

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

label_encoder = LabelEncoder()
df["zone_encoded"] = label_encoder.fit_transform(df["LOTISSEMENT"])

print("Zones connues par l'encodeur :", list(label_encoder.classes_))

X = df[["SUPERFICIE_M2", "zone_encoded", "infrastructure"]].rename(
    columns={"SUPERFICIE_M2": "superficie"}
)
y = df["PRIX-VENTE"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 4. Entraînement — Régression Linéaire Multiple

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

modele_lr = LinearRegression()
modele_lr.fit(X_train, y_train)

pred_lr = modele_lr.predict(X_test)
print("R² (LR) :", round(r2_score(y_test, pred_lr), 3))
print("MAE (LR) :", round(mean_absolute_error(y_test, pred_lr), 2))

## 5. Entraînement — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

modele_rf = RandomForestRegressor(n_estimators=100, random_state=42)
modele_rf.fit(X_train, y_train)

pred_rf = modele_rf.predict(X_test)
print("R² (RF) :", round(r2_score(y_test, pred_rf), 3))
print("MAE (RF) :", round(mean_absolute_error(y_test, pred_rf), 2))

## 6. Entraînement — Isolation Forest (détection de fraude / anomalies)

⚠️ Comme la colonne `FRAUDE` du dataset ne contient quasiment que des 0, on ne peut pas *valider* l'Isolation Forest avec des vrais cas de fraude connus. On l'entraîne de façon non supervisée sur l'ensemble des features, pour détecter les parcelles dont les caractéristiques (superficie / zone / prix implicite) sortent du lot.

In [ ]:
from sklearn.ensemble import IsolationForest

# On entraîne sur superficie, zone_encoded, infrastructure ET le prix,
# car une anomalie de prix par rapport à la superficie/zone est le signal recherché.
X_fraude = df[["SUPERFICIE_M2", "zone_encoded", "infrastructure", "PRIX-VENTE"]]

modele_fraude = IsolationForest(contamination=0.05, random_state=42)
modele_fraude.fit(X_fraude)

predictions = modele_fraude.predict(X_fraude)
n_anomalies = (predictions == -1).sum()
print(f"Anomalies détectées : {n_anomalies} / {len(df)} ({n_anomalies/len(df)*100:.1f}%)")

## 7. Export des modèles (.pkl)

Les fichiers seront visibles dans l'onglet **Output** de Kaggle après exécution, et téléchargeables individuellement.

In [ ]:
import joblib

joblib.dump(modele_lr, "/kaggle/working/modele_lr.pkl")
joblib.dump(modele_rf, "/kaggle/working/modele_rf.pkl")
joblib.dump(modele_fraude, "/kaggle/working/modele_fraude.pkl")
joblib.dump(label_encoder, "/kaggle/working/label_encoder.pkl")

print("Modèles exportés dans /kaggle/working/")

## 8. Test rapide d'une prédiction

In [ ]:
def estimer_prix(superficie, zone, infrastructure):
    zone_encoded = label_encoder.transform([zone.lower()])[0]
    X_input = np.array([[superficie, zone_encoded, infrastructure]])
    prix_lr = modele_lr.predict(X_input)[0]
    prix_rf = modele_rf.predict(X_input)[0]
    return {
        "prix_lr": round(float(prix_lr), 2),
        "prix_rf": round(float(prix_rf), 2),
        "prix_moyen": round(float((prix_lr + prix_rf) / 2), 2),
    }

print(estimer_prix(200, "mapendo", 1))